# Proyecto 1

## Diagnóstico inicial y plan de limpieza

Las 17 variables del conjunto: `CODIGO, DISTRITO, DEPARTAMENTO, MUNICIPIO, ESTABLECIMIENTO, DIRECCION, TELEFONO,
SUPERVISOR, DIRECTOR, NIVEL, SECTOR, AREA, STATUS, MODALIDAD, JORNADA, PLAN, DEPARTAMENTAL`.


In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 60)

RAW = Path("data/diversificado_consolidado.csv")

## 1–2. Carga de los datos crudos (descargados y guardados en .csv)

In [2]:
def cargar(path):
    for enc in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            return pd.read_csv(path, dtype=str, keep_default_na=False, encoding=enc)
        except (UnicodeDecodeError, UnicodeError):
            continue
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding="latin-1")

df = cargar(RAW)
print(f"Registros: {len(df):,}   Variables: {df.shape[1]}")
df.head()

Registros: 11,867   Variables: 17


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
3,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,78232301,JOSE ARTURO CHOC CHEN,VIRGINIA SOLANO SERRANO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
4,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,MOISÉS ADRIÁN LÓPEZ PÉREZ,ESTELA DEL CARMEN QUIM ROSALES,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ


### Utilidades
Tratamos como **faltante** las celdas vacías y los marcadores de ausencia (`--`, `---`, `-`, `N/A`), que el portal usa en lugar de nulos reales.

In [3]:
PLACEHOLDERS = {"", "nan", "none", "null", "n/a", "-", "--", "---", "----", "s/n", "sin dato"}

def es_faltante(serie):
    s = serie.astype(str).str.strip().str.lower()
    return s.isin(PLACEHOLDERS)

def sin_tildes(s):
    s = unicodedata.normalize("NFKD", str(s))
    return "".join(ch for ch in s if not unicodedata.combining(ch))

def buscar_col(*claves):
    for col in df.columns:
        cl = sin_tildes(col).lower()
        if all(k in cl for k in claves):
            return col
    return None

faltante_mask = pd.DataFrame({col: es_faltante(df[col]) for col in df.columns})

## 3. Diagnóstico del estado inicial

### a) Número de registros y variables

In [4]:
print(f"Registros (filas): {df.shape[0]:,}")
print(f"Variables (columnas): {df.shape[1]}")
pd.DataFrame({"variable": df.columns})

Registros (filas): 11,867
Variables (columnas): 17


,variable
0,CODIGO
1,DISTRITO
2,DEPARTAMENTO
3,MUNICIPIO
4,ESTABLECIMIENTO
5,DIRECCION
6,TELEFONO
7,SUPERVISOR
8,DIRECTOR
9,NIVEL


### b) Tipo de dato de cada variable
Todo se cargó como texto para preservar ceros a la izquierda (códigos, teléfonos). Aquí se infiere además el **tipo semántico** observando el contenido.

In [5]:
def tipo_semantico(serie):
    v = serie[~es_faltante(serie)].astype(str).str.strip()
    if v.empty:
        return "vacía"
    if v.str.fullmatch(r"\d{2}-\d{2}-\d{4}-\d{2}").mean() > 0.8:
        return "código establecimiento (##-##-####-##)"
    if v.str.fullmatch(r"\d{2}-\d{2,3}(-\d{3,4})?").mean() > 0.6:
        return "código distrito"
    if v.str.fullmatch(r"[-+()\d\s/]{6,}").mean() > 0.7:
        return "teléfono"
    if v.nunique() <= max(30, 0.02 * len(v)):
        return "categórica"
    return "texto libre"

tipos = pd.DataFrame({
    "dtype_pandas": df.dtypes.astype(str),
    "tipo_semantico": {col: tipo_semantico(df[col]) for col in df.columns},
    "ejemplo": {col: next((x for x in df[col] if str(x).strip() not in PLACEHOLDERS), "") for col in df.columns},
})
tipos

,dtype_pandas,tipo_semantico,ejemplo
CODIGO,str,código establecimiento (##-##-####-##),16-01-0137-46
DISTRITO,str,código distrito,16-006
DEPARTAMENTO,str,categórica,ALTA VERAPAZ
MUNICIPIO,str,texto libre,COBAN
ESTABLECIMIENTO,str,texto libre,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN
DIRECCION,str,texto libre,6A. AVENIDA 1-15 ZONA 4
TELEFONO,str,teléfono,77945104
SUPERVISOR,str,texto libre,JORGE EDUARDO PAQUE LAZARO
DIRECTOR,str,texto libre,GUSTAVO ADOLFO SIERRA POP
NIVEL,str,categórica,DIVERSIFICADO


### c) Cantidad y porcentaje de valores faltantes por variable

In [6]:
faltantes = pd.DataFrame({
    "n_faltantes": faltante_mask.sum(),
    "pct_faltantes": (faltante_mask.mean() * 100).round(2),
}).sort_values("pct_faltantes", ascending=False)
faltantes

,n_faltantes,pct_faltantes
DIRECTOR,2024,17.06
TELEFONO,946,7.97
SUPERVISOR,535,4.51
DISTRITO,532,4.48
DIRECCION,80,0.67
ESTABLECIMIENTO,5,0.04
AREA,0,0.00
PLAN,0,0.00
JORNADA,0,0.00
MODALIDAD,0,0.00


### d) Cantidad de valores únicos por variable

In [7]:
unicos = pd.DataFrame({
    "n_unicos": {col: df.loc[~es_faltante(df[col]), col].str.strip().nunique() for col in df.columns},
})
unicos["n_registros"] = len(df)
unicos["ratio_unicidad"] = (unicos["n_unicos"] / unicos["n_registros"]).round(3)
unicos.sort_values("n_unicos", ascending=False)

,n_unicos,n_registros,ratio_unicidad
CODIGO,11867,11867,1.000
DIRECCION,7437,11867,0.627
TELEFONO,6570,11867,0.554
ESTABLECIMIENTO,6312,11867,0.532
DIRECTOR,5512,11867,0.464
DISTRITO,1681,11867,0.142
SUPERVISOR,1280,11867,0.108
MUNICIPIO,352,11867,0.030
DEPARTAMENTAL,26,11867,0.002
DEPARTAMENTO,23,11867,0.002


### e) Registros duplicados exactos
Además del duplicado exacto (fila idéntica en las 17 columnas) reportamos el **duplicado por clave de negocio**: un mismo `CODIGO` puede repetirse legítimamente con distinta jornada/plan, así que se mide aparte.

In [8]:
col_cod = buscar_col("codigo")

dup_exactos = df.duplicated(keep="first").sum()
print(f"Filas duplicadas EXACTAS (todas las columnas): {dup_exactos}")

if col_cod:
    dup_cod = df[~es_faltante(df[col_cod])].duplicated(subset=[col_cod], keep=False).sum()
    print(f"Filas con CODIGO repetido (clave de negocio): {dup_cod}")
    clave = [c for c in [col_cod, buscar_col('jorn'), buscar_col('plan')] if c]
    dup_clave = df.duplicated(subset=clave, keep=False).sum()
    print(f"Filas con ({', '.join(clave)}) repetida: {dup_clave}")

Filas duplicadas EXACTAS (todas las columnas): 0
Filas con CODIGO repetido (clave de negocio): 0
Filas con (CODIGO, JORNADA, PLAN) repetida: 0


### f) Valores fuera de dominio o inconsistentes

Se comparan las variables categóricas contra su **dominio esperado** (departamentos oficiales, sectores/niveles/áreas/modalidades del portal). Se comparan en MAYÚSCULAS sin tildes.

In [9]:
DEPARTAMENTOS = {
    "ALTA VERAPAZ","BAJA VERAPAZ","CHIMALTENANGO","CHIQUIMULA","EL PROGRESO","ESCUINTLA",
    "GUATEMALA","HUEHUETENANGO","IZABAL","JALAPA","JUTIAPA","PETEN","QUETZALTENANGO",
    "QUICHE","RETALHULEU","SACATEPEQUEZ","SAN MARCOS","SANTA ROSA","SOLOLA",
    "SUCHITEPEQUEZ","TOTONICAPAN","ZACAPA",
}
SECTORES   = {"OFICIAL","PRIVADO","MUNICIPAL","COOPERATIVA"}
NIVELES    = {"DIVERSIFICADO"}
AREAS      = {"URBANA","RURAL"}
MODALIDADES= {"MONOLINGUE","BILINGUE"}

def norm(x):
    return sin_tildes(str(x)).upper().strip()

def fuera_de_dominio(col, dominio):
    v = df.loc[~es_faltante(df[col]), col].map(norm)
    return v[~v.isin(dominio)].value_counts()

for etiqueta, claves, dom in [
    ("DEPARTAMENTO", ("depart",),  DEPARTAMENTOS),   # nota: 'DEPARTAMENTAL' también matchea; ver abajo
    ("SECTOR",       ("sector",),  SECTORES),
    ("NIVEL",        ("nivel",),   NIVELES),
    ("AREA",         ("area",),    AREAS),
    ("MODALIDAD",    ("modalidad",),MODALIDADES),
]:
    col = etiqueta if etiqueta in df.columns else buscar_col(*claves)
    if col and col in df.columns:
        fuera = fuera_de_dominio(col, dom)
        print(f"\n### {col}: {len(fuera)} valor(es) fuera de dominio")
        print(fuera.to_string() if len(fuera) else "  (todos dentro de dominio)")


### DEPARTAMENTO: 1 valor(es) fuera de dominio
DEPARTAMENTO
CIUDAD CAPITAL    2161

### SECTOR: 0 valor(es) fuera de dominio
  (todos dentro de dominio)

### NIVEL: 0 valor(es) fuera de dominio
  (todos dentro de dominio)

### AREA: 1 valor(es) fuera de dominio
AREA
SIN ESPECIFICAR    3

### MODALIDAD: 0 valor(es) fuera de dominio
  (todos dentro de dominio)


In [10]:
# Categóricas sin dominio cerrado fijo: se listan sus valores para revisión
for col in [c for c in ["STATUS","JORNADA","PLAN"] if c in df.columns]:
    print(f"\n### {col} — valores observados:")
    print(df.loc[~es_faltante(df[col]), col].map(str.strip).value_counts().to_string())


### STATUS — valores observados:
STATUS
ABIERTA                    6860
CERRADA TEMPORALMENTE      3006
CERRADA DEFINITIVAMENTE    1841
TEMPORAL TITULOS            110
TEMPORAL NOMBRAMIENTO        50

### JORNADA — valores observados:
JORNADA
DOBLE          3772
VESPERTINA     3407
MATUTINA       3045
SIN JORNADA    1099
NOCTURNA        415
INTERMEDIA      129

### PLAN — valores observados:
PLAN
DIARIO(REGULAR)                          7452
FIN DE SEMANA                            2898
SEMIPRESENCIAL (FIN DE SEMANA)            542
SEMIPRESENCIAL (UN DÍA A LA SEMANA)       437
A DISTANCIA                               188
SEMIPRESENCIAL                            118
VIRTUAL A DISTANCIA                        70
SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)      67
SABATINO                                   59
DOMINICAL                                  27
MIXTO                                       4
IRREGULAR                                   3
INTERCALADO                                 2


In [11]:
# Consistencia entre columnas: DEPARTAMENTO vs DEPARTAMENTAL, y prefijo de DISTRITO vs código
if "DEPARTAMENTO" in df.columns and "DEPARTAMENTAL" in df.columns:
    difs = (df["DEPARTAMENTO"].map(norm) != df["DEPARTAMENTAL"].map(norm)).sum()
    print(f"Filas donde DEPARTAMENTO != DEPARTAMENTAL: {difs}  (si es 0, DEPARTAMENTAL es redundante)")

col_dist = buscar_col("distrito")
if col_cod and col_dist:
    pref_cod  = df[col_cod].str[:2]
    pref_dist = df.loc[~es_faltante(df[col_dist]), col_dist].str[:2]
    inconsist = (df.loc[pref_dist.index, col_cod].str[:2] != pref_dist).sum()
    print(f"Filas donde el prefijo de DISTRITO != prefijo de CODIGO: {inconsist}")

Filas donde DEPARTAMENTO != DEPARTAMENTAL: 4130  (si es 0, DEPARTAMENTAL es redundante)
Filas donde el prefijo de DISTRITO != prefijo de CODIGO: 1958


### g) Formatos inconsistentes

Se detectan: espacios dobles/sobrantes, **mojibake** (caracteres corruptos como `¿`, `Ã`, comillas escapadas `\'`), formato irregular del `DISTRITO`, teléfonos con varios números o separadores, y **fechas incrustadas** en direcciones.

In [12]:
def con_espacios(s):   return (s.astype(str) != s.astype(str).str.strip()) | s.astype(str).str.contains(r"\s{2,}", regex=True, na=False)
def con_mojibake(s):   return s.astype(str).str.contains(r"[¿�]|Ã.|â€|Â.|\\['\"]", regex=True, na=False)
def con_fecha(s):      return s.astype(str).str.contains(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", regex=True, na=False)

fmt = pd.DataFrame({
    "espacios_extra":   {c: int(con_espacios(df[c]).sum())  for c in df.columns},
    "posible_mojibake": {c: int(con_mojibake(df[c]).sum())  for c in df.columns},
    "fecha_incrustada": {c: int(con_fecha(df[c]).sum())     for c in df.columns},
})
print(fmt[(fmt.T != 0).any()].to_string())

# Formato de CODIGO y DISTRITO
if col_cod:
    mal_cod = (~df[col_cod].str.match(r"^\d{2}-\d{2}-\d{4}-\d{2}$", na=False)).sum()
    print(f"\nCODIGO con formato != ##-##-####-##: {mal_cod}")
if col_dist:
    d = df.loc[~es_faltante(df[col_dist]), col_dist]
    print("Formatos de DISTRITO (conteo por patrón de longitud):")
    print(d.str.replace(r'\d', '#', regex=True).value_counts().to_string())

col_tel = buscar_col("telefono")
if col_tel:
    t = df.loc[~es_faltante(df[col_tel]), col_tel]
    n_tel_raro = (~t.str.fullmatch(r"\d{8}")).sum()
    print(f"\nTELEFONO con separadores o múltiples números (no 8 dígitos limpios): {n_tel_raro} de {len(t)}")

                 espacios_extra  posible_mojibake  fecha_incrustada
ESTABLECIMIENTO               0                 2                 0
DIRECCION                     0                 5                18
TELEFONO                      0                 0                 1
DIRECTOR                      0                 1                 0

CODIGO con formato != ##-##-####-##: 0
Formatos de DISTRITO (conteo por patrón de longitud):
DISTRITO
##-##-####    6226
##-###        5039
##-             70

TELEFONO con separadores o múltiples números (no 8 dígitos limpios): 251 de 10921


### h) Identificación de problemas potenciales de calidad — resumen

Consolidación por variable de todo lo anterior.

In [13]:
resumen = pd.DataFrame(index=df.columns)
resumen["tipo"]             = tipos["tipo_semantico"]
resumen["pct_faltantes"]    = faltantes["pct_faltantes"]
resumen["n_unicos"]         = unicos["n_unicos"]
resumen["espacios_extra"]   = fmt["espacios_extra"]
resumen["posible_mojibake"] = fmt["posible_mojibake"]
resumen["fecha_incrustada"] = fmt["fecha_incrustada"]
resumen

,tipo,pct_faltantes,n_unicos,espacios_extra,posible_mojibake,fecha_incrustada
CODIGO,código establecimiento (##-##-####-##),0.00,11867,0,0,0
DISTRITO,código distrito,4.48,1681,0,0,0
DEPARTAMENTO,categórica,0.00,23,0,0,0
MUNICIPIO,texto libre,0.00,352,0,0,0
ESTABLECIMIENTO,texto libre,0.04,6312,0,2,0
DIRECCION,texto libre,0.67,7437,0,5,18
TELEFONO,teléfono,7.97,6570,0,0,1
SUPERVISOR,texto libre,4.51,1280,0,0,0
DIRECTOR,texto libre,17.06,5512,0,1,0
NIVEL,categórica,0.00,1,0,0,0


**Hallazgos principales (resumen cualitativo):**

- `NIVEL` es constante (`DIVERSIFICADO`) por el filtro de la descarga: no aporta variabilidad.
- `TELEFONO`, `DIRECTOR`, `DISTRITO` y `DIRECCION` concentran los **faltantes**; `DIRECTOR` además usa marcadores `--`/`---`.
- `TELEFONO` mezcla formatos: 8 dígitos, con guiones, y hasta **dos números** en una celda.
- `DISTRITO` tiene **varios formatos** distintos.
- Textos (`ESTABLECIMIENTO`, `DIRECCION`, `SUPERVISOR`, `DIRECTOR`) con **espacios dobles, mojibake** (`¿`, comillas escapadas) y **tildes inconsistentes** (`SURÁM`/`SURAM`).
- Alguna `DIRECCION` trae una **fecha incrustada** (p. ej. `02/01/2018`), dato que no corresponde a la variable.
- `AREA` puede traer valores fuera de `{URBANA, RURAL}` (p. ej. `SIN ESPECIFICAR`).


## 4. Plan de limpieza

Para cada variable: **problema** encontrado, **regla** de corrección (y por qué funcionaría) y **riesgo** de la transformación. Es la guía que usará el equipo para ejecutar la limpieza después.

| Variable | Problema encontrado | Regla de corrección (y por qué funciona) | Riesgo |
|---|---|---|---|
| `DISTRITO` | Dos formatos (`16-006` y `16-01-0926`) y algunos faltantes. | Documentar ambos formatos; separar en “distrito corto/largo” solo si el análisis lo requiere; no imputar faltantes. | Medio: forzar un único formato podría destruir información; mejor dejar como texto y anotar. |
| `DEPARTAMENTO` | Mayúsculas/tildes; validar contra los 22 oficiales. | Normalizar a MAYÚSCULAS sin tildes y mapear al catálogo. Dominio cerrado y conocido. | Bajo. |
| `MUNICIPIO` | Tildes/espacios inconsistentes. | Normalizar (MAYÚSCULAS sin tildes, colapsar espacios) y validar contra municipios del departamento. | Medio: un municipio mal escrito podría no mapear; se deja marcado para revisión. |
| `ESTABLECIMIENTO` | Espacios dobles, comillas escapadas, mojibake, algún vacío. | `strip` + colapsar espacios; corregir codificación (`ftfy`); dejar vacío como faltante. | Medio: la corrección de mojibake mal aplicada puede alterar nombres correctos; revisar muestra. |
| `DIRECCION` | Espacios, mojibake y **fechas incrustadas** (`02/01/2018`). | Colapsar espacios; extraer/eliminar la fecha con regex `\d{1,2}/\d{1,2}/\d{2,4}` moviéndola a una nota. | Medio: una fecha podría ser parte legítima de la dirección (raro); revisar los casos detectados. |
| `TELEFONO` | Vacíos, guiones, y **dos números** en una celda. | Quedarse con dígitos; si hay 16 dígitos, separar en dos columnas; validar longitud 8 (GT); vacío = faltante. | Bajo/medio: números atípicos se marcan como inválidos, **no** se borran. |
| `SUPERVISOR` | Tildes/espacios inconsistentes. | `strip` + colapsar espacios; corregir codificación. No imputar. | Bajo. |
| `DIRECTOR` | Marcadores `--`/`---`/`-` y vacíos. | Convertir marcadores a faltante (NaN); `strip`; corregir codificación. | Bajo. |
| `NIVEL` | Constante `DIVERSIFICADO`. | Verificar que todo sea `DIVERSIFICADO`; se puede conservar como control o descartar por nula varianza. | Bajo. |
| `SECTOR` | Dominio `{OFICIAL, PRIVADO, MUNICIPAL, COOPERATIVA}`. | Estandarizar contra el dominio cerrado. | Bajo. |
| `AREA` | Puede traer `SIN ESPECIFICAR` fuera de `{URBANA, RURAL}`. | Estandarizar; `SIN ESPECIFICAR` → categoría explícita o faltante, según criterio. | Bajo. |
| `STATUS` | Varias categorías (`ABIERTA`, `CERRADA…`, `TEMPORAL NOMBRAMIENTO`). | Estandarizar contra el conjunto observado; unificar variantes de escritura. | Bajo. |
| `MODALIDAD` | Dominio `{MONOLINGUE, BILINGUE}`. | Estandarizar contra el dominio. | Bajo. |
| `JORNADA` | `SIN JORNADA` y variantes. | Estandarizar; decidir si `SIN JORNADA` es faltante o categoría propia. | Bajo. |
| `PLAN` | Muchas variantes (`SEMIPRESENCIAL (FIN DE SEMANA)`, `VIRTUAL A DISTANCIA`, …). | Definir catálogo de planes y mapear variantes a categorías canónicas. | Medio: agrupar de más puede perder matices; documentar el mapeo. |
| `DEPARTAMENTAL` | Parece **redundante** con `DEPARTAMENTO`. | Confirmar igualdad fila a fila (inciso f); si coincide siempre, eliminar la columna. | Bajo, si se confirma la redundancia. |
| *Filas* | Duplicados exactos y por clave de negocio (mismo `CODIGO`, distinta jornada/plan). | Eliminar solo **duplicados exactos** con `drop_duplicates()`; los de clave de negocio se conservan (son servicios distintos). | Bajo: deduplicar por fila completa evita borrar registros legítimos. |


## 5. Limpieza del conjunto de datos

Ejecutamos el plan del punto 4 de forma **reproducible**: se parte del `DataFrame` crudo cargado
como texto y se documenta cada decisión. Trabajamos sobre una copia `dfc` para poder comparar
*antes vs. después*.

> **Corrección al plan (punto 4):** el diagnóstico anticipó que `DEPARTAMENTAL` podía ser *redundante*
> con `DEPARTAMENTO`. Al revisar sus valores se confirma que **no lo es**: `DEPARTAMENTAL` es la
> **Dirección Departamental de Educación (DIDEDUC)**, una unidad *administrativa*. El departamento
> Guatemala se divide en 4 direcciones (`GUATEMALA NORTE/SUR/ORIENTE/OCCIDENTE`) y Quiché en 2
> (`QUICHE`, `QUICHE NORTE`). Por eso se **conserva** y se usa sólo para verificar consistencia.
> Asimismo, `CIUDAD CAPITAL` en `DEPARTAMENTO` no es un departamento: es la Ciudad de Guatemala
> (municipio de Guatemala, departamento de Guatemala), identificada por el prefijo de código `00`.

### 5.a Valores faltantes, cadenas vacías, espacios y marcadores de ausencia

Regla única de faltante: una celda es faltante si —tras recortar espacios y pasar a minúsculas— queda
vacía o coincide con un **marcador de ausencia**. El portal MINEDUC usa `--`, `---`, `-`, `.`, `N/A`,
`S/N`, etc. en lugar de nulos reales; todos se convierten a `NA` (`np.nan`) para un tratamiento uniforme.

In [14]:
import re, unicodedata
import numpy as np
import pandas as pd
from rapidfuzz import fuzz

# Marcadores que representan AUSENCIA de dato (no valores legítimos)
PLACEHOLDERS = {"", "nan", "none", "null", "n/a", "na", "-", "--", "---", "----", "s/n", "sin dato", "."}

def sin_tildes(s):
    s = unicodedata.normalize("NFKD", str(s))
    return "".join(c for c in s if not unicodedata.combining(c))

def limpiar_texto(x):
    """Normaliza texto: NFC, quita caracteres invisibles, colapsa espacios, recorta;
    devuelve NA si el resultado es un marcador de ausencia."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return np.nan
    s = unicodedata.normalize("NFC", str(x))
    for inv in ("\u00a0", "\u200b", "\u200e", "\u200f", "\ufeff", "\t"):  # NBSP, ZWSP, LRM, RLM, BOM, tab
        s = s.replace(inv, " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = _quitar_comillas_envolventes(s)
    return np.nan if s.lower() in PLACEHOLDERS else s

def _quitar_comillas_envolventes(s):
    """Quita comillas dobles que envuelven TODO el valor (espurias del portal),
    conservando las comillas internas legítimas (p. ej. COLEGIO \"LA INMACULADA\")."""
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"' and s[1:-1].count('"') == 0:
        s = s[1:-1].strip()
    elif s.count('"') == 1 and (s.startswith('"') or s.endswith('"')):
        s = s.strip('"').strip()
    return s

def a_faltante(x):
    """Convierte marcadores de ausencia a NA sin alterar el resto (para códigos/teléfonos)."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return np.nan
    return np.nan if str(x).strip().lower() in PLACEHOLDERS else str(x)

dfc = df.copy()
faltantes_antes = df.apply(lambda s: s.str.strip().str.lower().isin(PLACEHOLDERS).sum())
print("Copia de trabajo `dfc`:", dfc.shape)

Copia de trabajo `dfc`: (11867, 17)


### 5.b–5.c Tipo de dato y normalización de texto

Aplicamos `limpiar_texto` a todas las variables de texto (recorta espacios iniciales/finales y
múltiples, elimina caracteres invisibles como NBSP/ZWSP/BOM, unifica a forma Unicode NFC y estandariza
mayúsculas/tildes en las categóricas). `CODIGO`, `DISTRITO` y `TELEFONO` se guardan **como texto**
para preservar ceros a la izquierda y separadores. Se verifica que las tildes legítimas y los apóstrofos
mayas (`K'AMOLB'E`) se conserven; **no hay mojibake real** (el diagnóstico lo reportó por un falso
positivo al confundir comillas/apóstrofos legítimos con caracteres corruptos).

También se retiran las **comillas dobles que envuelven todo el nombre** (`"INSTITUTO ... IMIF"` → sin
comillas), que son un artefacto del portal, **conservando** las comillas internas legítimas
(`COLEGIO "LA INMACULADA"`, `2A. CALLE "A"`).

In [15]:
COLS_TEXTO = ["DEPARTAMENTO","MUNICIPIO","ESTABLECIMIENTO","DIRECCION","SUPERVISOR",
              "DIRECTOR","NIVEL","SECTOR","AREA","STATUS","MODALIDAD","JORNADA","PLAN","DEPARTAMENTAL"]
for c in COLS_TEXTO:
    dfc[c] = dfc[c].map(limpiar_texto)

# Códigos y teléfono: sólo marcadores -> NA, sin tocar dígitos/guiones
dfc["CODIGO"]   = dfc["CODIGO"].map(a_faltante).map(lambda x: x.replace(" ","") if isinstance(x,str) else x)
dfc["DISTRITO"] = dfc["DISTRITO"].map(a_faltante).map(lambda x: x.replace(" ","") if isinstance(x,str) else x)
dfc["TELEFONO"] = dfc["TELEFONO"].map(a_faltante)

comp = pd.DataFrame({"faltantes_antes": faltantes_antes,
                     "faltantes_despues(NA)": dfc.isna().sum()})
comp["nuevos_NA"] = comp["faltantes_despues(NA)"] - comp["faltantes_antes"]
comp

,faltantes_antes,faltantes_despues(NA),nuevos_NA
CODIGO,0,0,0
DISTRITO,532,532,0
DEPARTAMENTO,0,0,0
MUNICIPIO,0,0,0
ESTABLECIMIENTO,5,5,0
DIRECCION,85,85,0
TELEFONO,946,946,0
SUPERVISOR,535,535,0
DIRECTOR,2038,2038,0
NIVEL,0,0,0


Las categóricas de dominio cerrado se estandarizan a MAYÚSCULAS sin tildes para comparar contra su
catálogo. `NIVEL` es constante (`DIVERSIFICADO`, por el filtro de descarga) y se conserva como control.

In [16]:
def estand(serie):
    return serie.map(lambda x: sin_tildes(x).upper().strip() if pd.notna(x) else x)

for c in ["SECTOR","NIVEL","MODALIDAD","AREA","STATUS","JORNADA","PLAN","DEPARTAMENTO","DEPARTAMENTAL"]:
    dfc[c] = estand(dfc[c])

for c in ["SECTOR","AREA","MODALIDAD","STATUS","JORNADA"]:
    print(f"{c:>10}: {sorted(dfc[c].dropna().unique())}")

    SECTOR: ['COOPERATIVA', 'MUNICIPAL', 'OFICIAL', 'PRIVADO']
      AREA: ['RURAL', 'SIN ESPECIFICAR', 'URBANA']
 MODALIDAD: ['BILINGUE', 'MONOLINGUE']
    STATUS: ['ABIERTA', 'CERRADA DEFINITIVAMENTE', 'CERRADA TEMPORALMENTE', 'TEMPORAL NOMBRAMIENTO', 'TEMPORAL TITULOS']
   JORNADA: ['DOBLE', 'INTERMEDIA', 'MATUTINA', 'NOCTURNA', 'SIN JORNADA', 'VESPERTINA']


### 5.d Consistencia de categorías

Las variables de dominio cerrado ya quedan en una única forma canónica (MAYÚSCULAS, sin tildes, sin
espacios sobrantes). Verificamos contra el catálogo oficial y, si aparece algún valor fuera de dominio,
lo resolvemos con similitud de cadenas (RapidFuzz) al valor canónico más parecido — sólo si la
similitud es alta; de lo contrario se marca para revisión.

In [17]:
DEPTOS_OFICIALES = {"ALTA VERAPAZ","BAJA VERAPAZ","CHIMALTENANGO","CHIQUIMULA","EL PROGRESO",
    "ESCUINTLA","GUATEMALA","HUEHUETENANGO","IZABAL","JALAPA","JUTIAPA","PETEN","QUETZALTENANGO",
    "QUICHE","RETALHULEU","SACATEPEQUEZ","SAN MARCOS","SANTA ROSA","SOLOLA","SUCHITEPEQUEZ",
    "TOTONICAPAN","ZACAPA"}
DOMINIOS = {
    "SECTOR":    {"OFICIAL","PRIVADO","MUNICIPAL","COOPERATIVA"},
    "NIVEL":     {"DIVERSIFICADO"},
    "AREA":      {"URBANA","RURAL"},
    "MODALIDAD": {"MONOLINGUE","BILINGUE"},
}
for col, dom in DOMINIOS.items():
    fuera = sorted(set(dfc[col].dropna().unique()) - dom)
    print(f"{col:>10}: fuera de dominio -> {fuera if fuera else 'ninguno'}")

# AREA: 'SIN ESPECIFICAR' no pertenece a {URBANA, RURAL} -> se trata como faltante
n_area = (dfc["AREA"]=="SIN ESPECIFICAR").sum()
dfc.loc[dfc["AREA"]=="SIN ESPECIFICAR","AREA"] = np.nan
print(f"\nAREA='SIN ESPECIFICAR' reclasificadas a NA: {n_area}")

    SECTOR: fuera de dominio -> ninguno
     NIVEL: fuera de dominio -> ninguno
      AREA: fuera de dominio -> ['SIN ESPECIFICAR']
 MODALIDAD: fuera de dominio -> ninguno

AREA='SIN ESPECIFICAR' reclasificadas a NA: 3


`STATUS`, `JORNADA` y `PLAN` no tienen un catálogo oficial cerrado, pero sus valores ya son
consistentes tras la normalización (una sola grafía por categoría). Se listan para dejar constancia;
no se fuerza ninguna unificación adicional para no perder matices (p. ej. los distintos tipos de
`SEMIPRESENCIAL` del `PLAN`).

In [18]:
for c in ["STATUS","JORNADA","PLAN"]:
    print(f"\n### {c}")
    print(dfc[c].value_counts(dropna=False).to_string())


### STATUS
STATUS
ABIERTA                    6860
CERRADA TEMPORALMENTE      3006
CERRADA DEFINITIVAMENTE    1841
TEMPORAL TITULOS            110
TEMPORAL NOMBRAMIENTO        50

### JORNADA
JORNADA
DOBLE          3772
VESPERTINA     3407
MATUTINA       3045
SIN JORNADA    1099
NOCTURNA        415
INTERMEDIA      129

### PLAN
PLAN
DIARIO(REGULAR)                          7452
FIN DE SEMANA                            2898
SEMIPRESENCIAL (FIN DE SEMANA)            542
SEMIPRESENCIAL (UN DIA A LA SEMANA)       437
A DISTANCIA                               188
SEMIPRESENCIAL                            118
VIRTUAL A DISTANCIA                        70
SEMIPRESENCIAL (DOS DIAS A LA SEMANA)      67
SABATINO                                   59
DOMINICAL                                  27
MIXTO                                       4
IRREGULAR                                   3
INTERCALADO                                 2


### 5.e / 5.f Formatos y valores inválidos

**`CODIGO`** debe cumplir `##-##-####-##`. Su prefijo (2 primeros dígitos) codifica el departamento y
es la clave más confiable del registro.

In [19]:
mal_cod = (~dfc["CODIGO"].fillna("").str.match(r"^\d{2}-\d{2}-\d{4}-\d{2}$")).sum()
print(f"CODIGO con formato != ##-##-####-##: {mal_cod}   (únicos: {dfc['CODIGO'].nunique()} de {len(dfc)})")

# Mapa prefijo -> departamento oficial (00 = Ciudad Capital y 01 = resto => GUATEMALA)
PREF_DEPTO = {"00":"GUATEMALA","01":"GUATEMALA","02":"EL PROGRESO","03":"SACATEPEQUEZ",
  "04":"CHIMALTENANGO","05":"ESCUINTLA","06":"SANTA ROSA","07":"SOLOLA","08":"TOTONICAPAN",
  "09":"QUETZALTENANGO","10":"SUCHITEPEQUEZ","11":"RETALHULEU","12":"SAN MARCOS",
  "13":"HUEHUETENANGO","14":"QUICHE","15":"BAJA VERAPAZ","16":"ALTA VERAPAZ","17":"PETEN",
  "18":"IZABAL","19":"ZACAPA","20":"CHIQUIMULA","21":"JALAPA","22":"JUTIAPA"}
pref = dfc["CODIGO"].str[:2]
print("Prefijos fuera del catálogo oficial:", sorted(set(pref.dropna()) - set(PREF_DEPTO)) or "ninguno")

CODIGO con formato != ##-##-####-##: 0   (únicos: 11867 de 11867)
Prefijos fuera del catálogo oficial: ninguno


**`DISTRITO`** presenta dos formatos legítimos del portal (`16-006` corto y `16-01-0926` largo) más
un formato trunco `01-`. No se fuerza un formato único (destruiría información); se documenta el patrón
y los truncos se marcan como faltantes por no ser códigos válidos.

In [20]:
patrones = dfc["DISTRITO"].dropna().str.replace(r"\d","#",regex=True).value_counts()
print("Formatos de DISTRITO:")
print(patrones.to_string())
# '01-' y similares (prefijo-guion sin cuerpo) no son código válido -> NA
trunco = dfc["DISTRITO"].fillna("").str.match(r"^\d{2}-$")
print(f"\nDISTRITO truncos ('##-') reclasificados a NA: {int(trunco.sum())}")
dfc.loc[trunco, "DISTRITO"] = np.nan

Formatos de DISTRITO:
DISTRITO
##-##-####    6226
##-###        5039
##-             70

DISTRITO truncos ('##-') reclasificados a NA: 70


**`TELEFONO`** (Guatemala: 8 dígitos). Una celda puede traer varios números separados por `-`, `,`
o espacios. Regla: extraer todos los tokens de exactamente 8 dígitos; el primero es
`TELEFONO_PRINCIPAL` y los demás van a `TELEFONO_ADICIONALES`. Si la celda tenía dato pero **ningún**
token de 8 dígitos (p. ej. `22067`, `41724130-00000000`), el número es inválido: `TELEFONO_VALIDO=False`
y `TELEFONO_PRINCIPAL=NA`. **No se elimina** el registro; se conserva el original en `TELEFONO`.

In [21]:
def parse_tels(x):
    if not isinstance(x, str):
        return []
    toks = re.split(r"[^\d]+", x)
    return [t for t in toks if len(t) == 8 and t != "00000000"]

_tels = dfc["TELEFONO"].map(parse_tels)
dfc["TELEFONO_PRINCIPAL"]   = _tels.map(lambda l: l[0] if l else np.nan)
dfc["TELEFONO_ADICIONALES"] = _tels.map(lambda l: ";".join(l[1:]) if len(l) > 1 else np.nan)
dfc["N_TELEFONOS"]          = _tels.map(len).astype("int16")
_tenia = dfc["TELEFONO"].notna()
dfc["TELEFONO_VALIDO"] = np.where(~_tenia, np.nan, dfc["TELEFONO_PRINCIPAL"].notna())

print("Con teléfono principal válido:", int(dfc["TELEFONO_PRINCIPAL"].notna().sum()))
print("Con dato pero inválido (sin 8 dígitos):", int((_tenia & dfc["TELEFONO_PRINCIPAL"].isna()).sum()))
print("Con más de un teléfono:", int((dfc["N_TELEFONOS"] > 1).sum()))
dfc.loc[_tenia & dfc["TELEFONO_PRINCIPAL"].isna(), ["CODIGO","TELEFONO"]].head(8)

Con teléfono principal válido: 10809
Con dato pero inválido (sin 8 dígitos): 112
Con más de un teléfono: 119


,CODIGO,TELEFONO
242,16-07-0253-46,4085613
669,04-01-0128-46,783928
829,04-01-3069-46,8392658
830,04-01-3071-46,8392658
1340,00-01-0189-46,2517748 2384159
1417,00-01-0277-46,2202869-2202873
1613,00-01-0642-46,230835
1622,00-01-0656-46,2232068


### 5.h Consistencia entre variables

1. **`DEPARTAMENTO` (texto) vs. prefijo de `CODIGO`.** Se deriva `DEPARTAMENTO_OFICIAL` desde el prefijo
   (fuente más confiable) y se contrasta con el texto (contando `CIUDAD CAPITAL` como `GUATEMALA`).
2. **`DEPARTAMENTO_OFICIAL` vs. `DEPARTAMENTAL` (DIDEDUC).** Cada dirección administrativa debe pertenecer
   a su departamento geográfico (`GUATEMALA *`→GUATEMALA, `QUICHE NORTE`→QUICHE).
3. **Fecha incrustada en `DIRECCION`.** Algunas direcciones traen una fecha `dd/mm/aaaa` (artefacto de
   captura). Se extrae a `FECHA_EN_DIRECCION` y se remueve de la dirección.

In [22]:
# 1) DEPARTAMENTO texto vs prefijo de CODIGO
dfc["DEPARTAMENTO_OFICIAL"] = pref.map(PREF_DEPTO)
dfc["ES_CIUDAD_CAPITAL"]    = (pref == "00")
dep_txt = dfc["DEPARTAMENTO"].map(lambda x: "GUATEMALA" if x == "CIUDAD CAPITAL" else x)
inc_dep = (dep_txt != dfc["DEPARTAMENTO_OFICIAL"]).sum()
print(f"Filas donde DEPARTAMENTO(texto) contradice el prefijo de CODIGO: {inc_dep}")

# 2) DIDEDUC (DEPARTAMENTAL) -> departamento geográfico esperado
def didedduc_a_depto(v):
    if pd.isna(v): return np.nan
    if v.startswith("GUATEMALA"): return "GUATEMALA"
    if v.startswith("QUICHE"):    return "QUICHE"
    return v
map_didec = dfc["DEPARTAMENTAL"].map(didedduc_a_depto)
inc_did = (map_didec != dfc["DEPARTAMENTO_OFICIAL"]).sum()
print(f"Filas donde DEPARTAMENTAL (DIDEDUC) no corresponde al departamento del CODIGO: {inc_did}")

# 3) Fecha incrustada en DIRECCION
FECHA_RE = re.compile(r"\b(\d{1,2}/\d{1,2}/\d{2,4})\b")
dfc["FECHA_EN_DIRECCION"] = dfc["DIRECCION"].map(
    lambda x: FECHA_RE.search(x).group(1) if isinstance(x, str) and FECHA_RE.search(x) else np.nan)
dfc["DIRECCION"] = dfc["DIRECCION"].map(
    lambda x: re.sub(r"\s+", " ", FECHA_RE.sub("", x)).strip() if isinstance(x, str) else x)
dfc["DIRECCION"] = dfc["DIRECCION"].replace("", np.nan)
print(f"Fechas extraídas de DIRECCION a FECHA_EN_DIRECCION: {int(dfc['FECHA_EN_DIRECCION'].notna().sum())}")

Filas donde DEPARTAMENTO(texto) contradice el prefijo de CODIGO: 0
Filas donde DEPARTAMENTAL (DIDEDUC) no corresponde al departamento del CODIGO: 0
Fechas extraídas de DIRECCION a FECHA_EN_DIRECCION: 18


### 5.g Registros duplicados (exactos y parciales)

**Exactos.** No hay filas idénticas ni `CODIGO` repetido: `CODIGO` es clave única (11 867 valores).

**Parciales.** Bloqueando por (departamento del código, municipio) y comparando `ESTABLECIMIENTO` y
`DIRECCION` con `token_sort_ratio` (RapidFuzz), buscamos pares muy similares. Un mismo edificio aparece
varias veces porque MINEDUC asigna un `CODIGO` distinto por **jornada/plan/modalidad**. Por eso
clasificamos los pares candidatos en:

- **Legítimos:** difieren en `JORNADA/PLAN/SECTOR/MODALIDAD/STATUS` → son *servicios distintos*, se conservan.
- **Sospechosos:** idénticos en todos esos campos y sólo cambia `CODIGO` → se **marcan** para revisión
  manual (`DUP_PARCIAL_SOSPECHOSO=True`). **No se eliminan automáticamente**: un código oficial distinto
  suele implicar un registro legal distinto.

In [23]:
print("Duplicados exactos (todas las columnas):", int(df.duplicated().sum()))
print("CODIGO duplicado:", int(dfc["CODIGO"].duplicated().sum()))

def clave_fuzzy(s):
    return sin_tildes(str(s)).upper().strip()
def perfil(r):
    return (dfc.at[r,"JORNADA"], dfc.at[r,"PLAN"], dfc.at[r,"SECTOR"],
            dfc.at[r,"MODALIDAD"], dfc.at[r,"STATUS"])

pares_legit, pares_susp, susp_idx = 0, 0, set()
for _, g in dfc.groupby([pref, dfc["MUNICIPIO"]], observed=True):
    if len(g) < 2:
        continue
    idx = g.index.tolist()
    nom = [clave_fuzzy(x) for x in g["ESTABLECIMIENTO"]]
    dire = [clave_fuzzy(x) for x in g["DIRECCION"]]
    for i in range(len(idx)):
        for j in range(i+1, len(idx)):
            if not (nom[i] and nom[j] and dire[i] and dire[j]):
                continue
            if fuzz.token_sort_ratio(nom[i], nom[j]) >= 90 and fuzz.token_sort_ratio(dire[i], dire[j]) >= 90:
                if perfil(idx[i]) == perfil(idx[j]):
                    pares_susp += 1
                    susp_idx.update([idx[i], idx[j]])
                else:
                    pares_legit += 1

dfc["DUP_PARCIAL_SOSPECHOSO"] = dfc.index.isin(susp_idx)
print(f"\nPares candidatos legítimos (distinta jornada/plan): {pares_legit}")
print(f"Pares candidatos SOSPECHOSOS (idénticos salvo CODIGO): {pares_susp}")
print(f"Registros marcados para revisión manual: {int(dfc['DUP_PARCIAL_SOSPECHOSO'].sum())}")
dfc.loc[dfc["DUP_PARCIAL_SOSPECHOSO"], ["CODIGO","ESTABLECIMIENTO","DIRECCION","JORNADA","PLAN"]].sort_values("ESTABLECIMIENTO").head(10)

Duplicados exactos (todas las columnas): 0
CODIGO duplicado: 0

Pares candidatos legítimos (distinta jornada/plan): 6390
Pares candidatos SOSPECHOSOS (idénticos salvo CODIGO): 568
Registros marcados para revisión manual: 1025


,CODIGO,ESTABLECIMIENTO,DIRECCION,JORNADA,PLAN
4703,01-08-0346-46,ACADEMIA COMERCIAL E INSTITUTO LOURDES,"5A. AVENIDA 6-51 ""A"" ZONA 1",DOBLE,FIN DE SEMANA
4704,01-08-0348-46,ACADEMIA COMERCIAL E INSTITUTO LOURDES,"5A. AVENIDA 6-51 ""A"" ZONA 1",DOBLE,FIN DE SEMANA
565,15-03-0778-46,CENTRO CULTURAL DE AMERICA,3A. CALLE 3-64 ZONA 2,VESPERTINA,DIARIO(REGULAR)
568,15-03-1888-46,CENTRO CULTURAL DE AMERICA,3A. CALLE 3-64 ZONA 2,VESPERTINA,DIARIO(REGULAR)
1793,00-01-2905-46,CENTRO DE COMPUTACION IMB-PC,14 CALLE 2-04 Y 13 CALLE 2-08,MATUTINA,DIARIO(REGULAR)
1840,00-01-4388-46,CENTRO DE COMPUTACION IMB-PC,14 CALLE 2-04 Y 13 CALLE 2-08,MATUTINA,DIARIO(REGULAR)
5269,01-10-0286-46,CENTRO DE COMPUTACIÓN CEI-PC LINDA VISTA,"LOTE 19 MANZANA A, LINDA VISTA I",DOBLE,SEMIPRESENCIAL
5270,01-10-0287-46,CENTRO DE COMPUTACIÓN CEI-PC LINDA VISTA,"LOTE 19 MANZANA A, LINDA VISTA I",DOBLE,SEMIPRESENCIAL
5266,01-10-0279-46,CENTRO DE COMPUTACIÓN CEI-PC VILLAS,LOTE 14 MANZANA 61 VILLAS DEL QUETZAL,DOBLE,SEMIPRESENCIAL
5271,01-10-0292-46,CENTRO DE COMPUTACIÓN CEI-PC VILLAS,LOTE 14 MANZANA 61 VILLAS DEL QUETZAL,DOBLE,SEMIPRESENCIAL


### 5.i Variables derivadas y libro de códigos

| Variable derivada | Cómo se calcula | Utilidad |
|---|---|---|
| `DEPARTAMENTO_OFICIAL` | Prefijo (2 díg.) de `CODIGO` → catálogo de 22 departamentos (00 y 01 = GUATEMALA). | Departamento geográfico confiable y validado; resuelve `CIUDAD CAPITAL`. |
| `ES_CIUDAD_CAPITAL` | `True` si el prefijo de `CODIGO` es `00`. | Distingue la Ciudad de Guatemala del resto del departamento. |
| `TELEFONO_PRINCIPAL` | Primer token de 8 dígitos extraído de `TELEFONO`. | Teléfono en formato uniforme y utilizable. |
| `TELEFONO_ADICIONALES` | Resto de tokens de 8 dígitos, unidos por `;`. | Conserva números secundarios sin perder información. |
| `N_TELEFONOS` | Cantidad de teléfonos válidos en la celda. | Métrica de contactabilidad. |
| `TELEFONO_VALIDO` | `True/False` si el registro con dato produjo ≥1 número válido (NA si no había dato). | Marca calidad sin borrar el original. |
| `FECHA_EN_DIRECCION` | Fecha `dd/mm/aaaa` extraída de `DIRECCION`. | Aísla un dato mal ubicado; deja `DIRECCION` limpia. |
| `DUP_PARCIAL_SOSPECHOSO` | `True` si el registro integra un par casi idéntico (nombre y dirección ≥90) sin diferencia de jornada/plan. | Señala posibles duplicados para revisión manual. |

Todas las variables originales conservan su significado; estas se **agregan**, no reemplazan.

In [24]:
# Tipos de dato finales (cierra el punto 5.b): categóricas, enteros y booleanas explícitas
CATEGORICAS = ["DEPARTAMENTO","MUNICIPIO","DEPARTAMENTO_OFICIAL","DEPARTAMENTAL",
               "SECTOR","AREA","STATUS","MODALIDAD","JORNADA","PLAN","NIVEL"]
for c in CATEGORICAS:
    dfc[c] = dfc[c].astype("category")
dfc["N_TELEFONOS"]            = dfc["N_TELEFONOS"].astype("int16")
dfc["ES_CIUDAD_CAPITAL"]      = dfc["ES_CIUDAD_CAPITAL"].astype("bool")
dfc["DUP_PARCIAL_SOSPECHOSO"] = dfc["DUP_PARCIAL_SOSPECHOSO"].astype("bool")
dfc["TELEFONO_VALIDO"]        = dfc["TELEFONO_VALIDO"].astype("boolean")  # nullable (admite NA)

tipos_final = pd.DataFrame({
    "dtype": dfc.dtypes.astype(str),
    "n_no_nulos": dfc.notna().sum(),
    "pct_faltantes": (dfc.isna().mean()*100).round(2),
    "n_unicos": dfc.nunique(dropna=True),
})
print("Dimensiones finales:", dfc.shape)
tipos_final

Dimensiones finales: (11867, 25)


,dtype,n_no_nulos,pct_faltantes,n_unicos
CODIGO,str,11867,0.00,11867
DISTRITO,str,11265,5.07,1678
DEPARTAMENTO,category,11867,0.00,23
MUNICIPIO,category,11867,0.00,352
ESTABLECIMIENTO,str,11862,0.04,6297
DIRECCION,str,11779,0.74,7425
TELEFONO,str,10921,7.97,6570
SUPERVISOR,str,11332,4.51,1280
DIRECTOR,str,9829,17.17,5511
NIVEL,category,11867,0.00,1


### Guardado del conjunto limpio y del libro de códigos

Se exporta `diversificado_limpio.csv` (UTF-8 con BOM para Excel) y `libro_de_codigos.csv`. El proceso es
**reproducible**: parte del crudo, aplica reglas deterministas y no depende de edición manual.

In [25]:
from pathlib import Path
OUT = Path("data"); OUT.mkdir(exist_ok=True)
dfc.to_csv(OUT/"diversificado_limpio.csv", index=False, encoding="utf-8-sig")

libro = pd.DataFrame([
 ("CODIGO","texto","Código único del establecimiento (##-##-####-##). Prefijo = departamento.","Sí (clave)"),
 ("DISTRITO","texto","Código de distrito escolar (formato corto o largo del portal).","No"),
 ("DEPARTAMENTO","categórica","Departamento según el portal (incluye 'CIUDAD CAPITAL').","No"),
 ("MUNICIPIO","texto","Municipio; en la capital corresponde a la ZONA.","No"),
 ("ESTABLECIMIENTO","texto","Nombre del establecimiento.","No"),
 ("DIRECCION","texto","Dirección física (sin fecha incrustada).","No"),
 ("TELEFONO","texto","Teléfono(s) tal como venían (original).","No"),
 ("SUPERVISOR","texto","Nombre del supervisor.","No"),
 ("DIRECTOR","texto","Nombre del director.","No"),
 ("NIVEL","categórica","Nivel escolar; constante 'DIVERSIFICADO'.","No"),
 ("SECTOR","categórica","OFICIAL/PRIVADO/MUNICIPAL/COOPERATIVA.","No"),
 ("AREA","categórica","URBANA/RURAL (NA si venía 'SIN ESPECIFICAR').","No"),
 ("STATUS","categórica","Estado del establecimiento.","No"),
 ("MODALIDAD","categórica","MONOLINGUE/BILINGUE.","No"),
 ("JORNADA","categórica","Jornada de estudio.","No"),
 ("PLAN","categórica","Plan de estudios.","No"),
 ("DEPARTAMENTAL","categórica","Dirección Departamental de Educación (DIDEDUC), administrativa.","No"),
 ("DEPARTAMENTO_OFICIAL","categórica (derivada)","Departamento geográfico validado desde el prefijo de CODIGO.","No"),
 ("ES_CIUDAD_CAPITAL","booleana (derivada)","True si el establecimiento está en la Ciudad de Guatemala (prefijo 00).","No"),
 ("FECHA_EN_DIRECCION","texto (derivada)","Fecha dd/mm/aaaa extraída de DIRECCION.","No"),
 ("TELEFONO_PRINCIPAL","texto (derivada)","Primer teléfono válido de 8 dígitos.","No"),
 ("TELEFONO_ADICIONALES","texto (derivada)","Teléfonos secundarios separados por ';'.","No"),
 ("N_TELEFONOS","entero (derivada)","Cantidad de teléfonos válidos.","No"),
 ("TELEFONO_VALIDO","booleana (derivada)","Calidad del teléfono (NA si no había dato).","No"),
 ("DUP_PARCIAL_SOSPECHOSO","booleana (derivada)","Marca de posible duplicado parcial para revisión manual.","No"),
], columns=["variable","tipo","descripcion","clave"])
libro.to_csv(OUT/"libro_de_codigos.csv", index=False, encoding="utf-8-sig")
print("Guardado: data/diversificado_limpio.csv  y  data/libro_de_codigos.csv")
libro

Guardado: data/diversificado_limpio.csv  y  data/libro_de_codigos.csv


,variable,tipo,descripcion,clave
0,CODIGO,texto,Código único del establecimiento (##-##-####-##). Prefij...,Sí (clave)
1,DISTRITO,texto,Código de distrito escolar (formato corto o largo del po...,No
2,DEPARTAMENTO,categórica,Departamento según el portal (incluye 'CIUDAD CAPITAL').,No
3,MUNICIPIO,texto,Municipio; en la capital corresponde a la ZONA.,No
4,ESTABLECIMIENTO,texto,Nombre del establecimiento.,No
5,DIRECCION,texto,Dirección física (sin fecha incrustada).,No
6,TELEFONO,texto,Teléfono(s) tal como venían (original).,No
7,SUPERVISOR,texto,Nombre del supervisor.,No
8,DIRECTOR,texto,Nombre del director.,No
9,NIVEL,categórica,Nivel escolar; constante 'DIVERSIFICADO'.,No


## 6. Registro de transformaciones

Bitácora de **todas** las modificaciones aplicadas en el punto 5. La columna *Registros afectados* se
**calcula en vivo** desde los datos (crudo `df` vs. limpio `dfc`), de modo que la tabla es reproducible
y siempre refleja los conteos reales.

In [26]:
# Conteos reales de cada transformación (recalculados desde df crudo y dfc limpio)
def _isph(serie):
    return serie.astype(str).str.strip().str.lower().isin(PLACEHOLDERS)
def _stnd(x):
    return sin_tildes(x).upper().strip()

ph_por_col = {c: int(_isph(df[c]).sum()) for c in
              ["DIRECTOR","TELEFONO","SUPERVISOR","DISTRITO","DIRECCION","ESTABLECIMIENTO"]}
n_ph_total   = sum(ph_por_col.values())
n_comillas   = int(df["ESTABLECIMIENTO"].str.match(r'^".*"$', na=False).sum())
n_tilde_dep  = int((df["DEPARTAMENTAL"].map(_stnd) != df["DEPARTAMENTAL"]).sum())
n_tilde_plan = int((df["PLAN"].map(_stnd) != df["PLAN"]).sum())
n_area       = int((df["AREA"].map(_stnd) == "SIN ESPECIFICAR").sum())
n_dist_trunc = int(df["DISTRITO"].str.match(r"^\d{2}-$", na=False).sum())
n_fecha      = int(dfc["FECHA_EN_DIRECCION"].notna().sum())
n_capital    = int((df["CODIGO"].str[:2] == "00").sum())
n_tel_dato   = int(dfc["TELEFONO"].notna().sum())
n_tel_mult   = int((dfc["N_TELEFONOS"] > 1).sum())
n_tel_inval  = int((dfc["TELEFONO"].notna() & dfc["TELEFONO_PRINCIPAL"].isna()).sum())
n_dup        = int(dfc["DUP_PARCIAL_SOSPECHOSO"].sum())

registro = pd.DataFrame([
 ("DIRECTOR, TELEFONO, SUPERVISOR, DISTRITO, DIRECCION, ESTABLECIMIENTO",
  "Marcadores de ausencia (--, ---, -, ., N/A, S/N) y cadenas vacías",
  "Convertir todos a NA",
  f"{n_ph_total} celdas (DIRECTOR {ph_por_col['DIRECTOR']}, TELEFONO {ph_por_col['TELEFONO']}, "
  f"SUPERVISOR {ph_por_col['SUPERVISOR']}, DISTRITO {ph_por_col['DISTRITO']}, "
  f"DIRECCION {ph_por_col['DIRECCION']}, ESTABLECIMIENTO {ph_por_col['ESTABLECIMIENTO']})",
  "Unifica el tratamiento de nulos; el portal usa marcadores en vez de nulos reales."),

 ("Todas las de texto",
  "Espacios iniciales/finales, múltiples, invisibles (NBSP/ZWSP/BOM) y forma Unicode",
  "strip + colapsar espacios + normalizar a NFC",
  "0 (los datos ya venían sin estos defectos; regla defensiva y reproducible)",
  "Garantiza texto uniforme sin depender de que el crudo esté limpio."),

 ("ESTABLECIMIENTO",
  "Comillas dobles que envuelven todo el nombre (artefacto del portal)",
  "Quitar comillas envolventes; conservar las internas legítimas",
  f"{n_comillas}",
  'Las comillas externas son espurias; las internas (COLEGIO "LA INMACULADA") sí tienen significado.'),

 ("DEPARTAMENTAL",
  "Tildes inconsistentes (p. ej. PETÉN, QUICHÉ)",
  "Estandarizar a MAYÚSCULAS sin tildes",
  f"{n_tilde_dep}",
  "Permite comparar categorías sin duplicados por acentuación."),

 ("PLAN",
  "Tildes inconsistentes (SEMIPRESENCIAL (UN DÍA...))",
  "Estandarizar a MAYÚSCULAS sin tildes",
  f"{n_tilde_plan}",
  "Unifica la grafía de cada plan sin perder sus variantes de contenido."),

 ("AREA",
  "Valor fuera de dominio 'SIN ESPECIFICAR' (dominio esperado: URBANA/RURAL)",
  "Reclasificar a NA",
  f"{n_area}",
  "'SIN ESPECIFICAR' equivale a ausencia de dato, no a una categoría válida."),

 ("DISTRITO",
  "Código trunco '##-' sin cuerpo (no es un distrito válido)",
  "Reclasificar a NA",
  f"{n_dist_trunc}",
  "Un prefijo sin número de distrito no identifica ninguna unidad."),

 ("DIRECCION",
  "Fecha dd/mm/aaaa incrustada en la dirección (dato mal ubicado)",
  "Extraer a FECHA_EN_DIRECCION y removerla del texto",
  f"{n_fecha}",
  "La fecha no pertenece al dominio de la dirección; se aísla sin perder el dato."),

 ("DEPARTAMENTO / CODIGO",
  "'CIUDAD CAPITAL' aparece como departamento pero no lo es",
  "Derivar DEPARTAMENTO_OFICIAL desde el prefijo del CODIGO (00 y 01 = GUATEMALA)",
  f"{n_capital} (Ciudad Capital reasignada a GUATEMALA)",
  "El prefijo del código es la fuente geográfica confiable; corrige el falso departamento."),

 ("TELEFONO",
  "Varios números en una celda, separadores mixtos y números inválidos (<8 dígitos)",
  "Extraer tokens de 8 dígitos -> TELEFONO_PRINCIPAL/_ADICIONALES/N_TELEFONOS/_VALIDO",
  f"{n_tel_dato} con dato ({n_tel_mult} con varios números, {n_tel_inval} inválidos marcados)",
  "Formato uniforme (8 díg. GT) sin borrar el original ni los registros inválidos."),

 ("Registros (filas)",
  "Posibles duplicados parciales (nombre y dirección ≥90) sin diferencia de jornada/plan",
  "Marcar DUP_PARCIAL_SOSPECHOSO = True (NO se eliminan)",
  f"{n_dup} filas marcadas",
  "Un CODIGO oficial distinto suele ser un registro legal distinto; se deja para revisión manual."),
], columns=["Variable","Problema detectado","Transformación","Registros afectados","Justificación"])

with pd.option_context("display.max_colwidth", None, "display.width", 240):
    from IPython.display import display
    display(registro)
registro.to_csv("data/registro_transformaciones.csv", index=False, encoding="utf-8-sig")
print("Guardado: data/registro_transformaciones.csv")

,Variable,Problema detectado,Transformación,Registros afectados,Justificación
0,"DIRECTOR, TELEFONO, SUPERVISOR, DISTRITO, DIRECCION, ESTABLECIMIENTO","Marcadores de ausencia (--, ---, -, ., N/A, S/N) y cadenas vacías",Convertir todos a NA,"4141 celdas (DIRECTOR 2038, TELEFONO 946, SUPERVISOR 535, DISTRITO 532, DIRECCION 85, ESTABLECIMIENTO 5)",Unifica el tratamiento de nulos; el portal usa marcadores en vez de nulos reales.
1,Todas las de texto,"Espacios iniciales/finales, múltiples, invisibles (NBSP/ZWSP/BOM) y forma Unicode",strip + colapsar espacios + normalizar a NFC,0 (los datos ya venían sin estos defectos; regla defensiva y reproducible),Garantiza texto uniforme sin depender de que el crudo esté limpio.
2,ESTABLECIMIENTO,Comillas dobles que envuelven todo el nombre (artefacto del portal),Quitar comillas envolventes; conservar las internas legítimas,70,"Las comillas externas son espurias; las internas (COLEGIO ""LA INMACULADA"") sí tienen significado."
3,DEPARTAMENTAL,"Tildes inconsistentes (p. ej. PETÉN, QUICHÉ)",Estandarizar a MAYÚSCULAS sin tildes,2025,Permite comparar categorías sin duplicados por acentuación.
4,PLAN,Tildes inconsistentes (SEMIPRESENCIAL (UN DÍA...)),Estandarizar a MAYÚSCULAS sin tildes,504,Unifica la grafía de cada plan sin perder sus variantes de contenido.
5,AREA,Valor fuera de dominio 'SIN ESPECIFICAR' (dominio esperado: URBANA/RURAL),Reclasificar a NA,3,"'SIN ESPECIFICAR' equivale a ausencia de dato, no a una categoría válida."
6,DISTRITO,Código trunco '##-' sin cuerpo (no es un distrito válido),Reclasificar a NA,70,Un prefijo sin número de distrito no identifica ninguna unidad.
7,DIRECCION,Fecha dd/mm/aaaa incrustada en la dirección (dato mal ubicado),Extraer a FECHA_EN_DIRECCION y removerla del texto,18,La fecha no pertenece al dominio de la dirección; se aísla sin perder el dato.
8,DEPARTAMENTO / CODIGO,'CIUDAD CAPITAL' aparece como departamento pero no lo es,Derivar DEPARTAMENTO_OFICIAL desde el prefijo del CODIGO (00 y 01 = GUATEMALA),2161 (Ciudad Capital reasignada a GUATEMALA),El prefijo del código es la fuente geográfica confiable; corrige el falso departamento.
9,TELEFONO,"Varios números en una celda, separadores mixtos y números inválidos (<8 dígitos)",Extraer tokens de 8 dígitos -> TELEFONO_PRINCIPAL/_ADICIONALES/N_TELEFONOS/_VALIDO,"10921 con dato (119 con varios números, 112 inválidos marcados)",Formato uniforme (8 díg. GT) sin borrar el original ni los registros inválidos.


Guardado: data/registro_transformaciones.csv


## 7. Validación del conjunto limpio

Pruebas **automáticas** que verifican la calidad del conjunto ya limpio (`dfc`). Cada prueba registra
`PASS`/`FAIL` con un detalle; al final se genera un reporte y se `assert`a que **todas** pasen, de modo
que ejecutar el notebook falla explícitamente si la calidad se degradara.

In [27]:
import pandas.api.types as ptypes

resultados = []
def check(nombre, condicion, detalle=""):
    resultados.append({"prueba": nombre, "estado": "PASS" if condicion else "FAIL", "detalle": detalle})

TEXTO = ["CODIGO","DISTRITO","ESTABLECIMIENTO","DIRECCION","TELEFONO","SUPERVISOR","DIRECTOR",
         "TELEFONO_PRINCIPAL","TELEFONO_ADICIONALES","FECHA_EN_DIRECCION"]

# ---- 1) No hay duplicados exactos ----
d_exact = int(dfc.duplicated().sum())
check("Sin duplicados exactos (fila completa)", d_exact == 0, f"{d_exact} duplicados")
d_cod = int(dfc["CODIGO"].duplicated().sum())
check("CODIGO es clave única", d_cod == 0, f"{d_cod} códigos repetidos")

# ---- 2) Sin espacios al inicio/final, dobles ni caracteres invisibles ----
def _texto(col):
    return dfc[col].dropna().astype(str)
mal_esp = {c: int(((_texto(c) != _texto(c).str.strip()) |
                    _texto(c).str.contains("\\s{2,}|[\u00a0\u200b\ufeff\t]", regex=True)).sum())
           for c in TEXTO + [c for c in dfc.columns if str(dfc[c].dtype)=="category"]}
tot_esp = sum(mal_esp.values())
_esp_bad = {k: v for k, v in mal_esp.items() if v}
check("Sin espacios inicio/final/dobles ni invisibles", tot_esp == 0,
      f"columnas con problema: {_esp_bad}")

# ---- 3) Teléfonos con formato consistente ----
tp = dfc["TELEFONO_PRINCIPAL"].dropna()
bad_tp = int((~tp.str.fullmatch(r"\d{8}")).sum())
tads = dfc["TELEFONO_ADICIONALES"].dropna().str.split(";").explode()
bad_ta = int((~tads.str.fullmatch(r"\d{8}")).sum()) if len(tads) else 0
coincide_n = bool((dfc["N_TELEFONOS"] == (dfc["TELEFONO_PRINCIPAL"].notna().astype(int) +
              dfc["TELEFONO_ADICIONALES"].fillna("").apply(lambda s: 0 if s=="" else s.count(";")+1))).all())
check("TELEFONO_PRINCIPAL: exactamente 8 dígitos", bad_tp == 0, f"{bad_tp} inválidos")
check("TELEFONO_ADICIONALES: cada número de 8 dígitos", bad_ta == 0, f"{bad_ta} inválidos")
check("N_TELEFONOS coincide con los teléfonos almacenados", coincide_n)

# ---- 4) Departamentos y municipios en su catálogo ----
CAT_DEP = DEPTOS_OFICIALES  # 22 oficiales (definido en 5.d)
DIDEDUC = {"ALTA VERAPAZ","BAJA VERAPAZ","CHIMALTENANGO","CHIQUIMULA","EL PROGRESO","ESCUINTLA",
    "GUATEMALA NORTE","GUATEMALA OCCIDENTE","GUATEMALA ORIENTE","GUATEMALA SUR","HUEHUETENANGO",
    "IZABAL","JALAPA","JUTIAPA","PETEN","QUETZALTENANGO","QUICHE","QUICHE NORTE","RETALHULEU",
    "SACATEPEQUEZ","SAN MARCOS","SANTA ROSA","SOLOLA","SUCHITEPEQUEZ","TOTONICAPAN","ZACAPA"}
fuera_dep_of = set(dfc["DEPARTAMENTO_OFICIAL"].dropna().astype(str)) - CAT_DEP
fuera_dep_pt = set(dfc["DEPARTAMENTO"].dropna().astype(str)) - (CAT_DEP | {"CIUDAD CAPITAL"})
fuera_didec  = set(dfc["DEPARTAMENTAL"].dropna().astype(str)) - DIDEDUC
check("DEPARTAMENTO_OFICIAL en catálogo de 22 departamentos", not fuera_dep_of, str(fuera_dep_of))
check("DEPARTAMENTO (portal) en catálogo (+ CIUDAD CAPITAL)", not fuera_dep_pt, str(fuera_dep_pt))
check("DEPARTAMENTAL en catálogo de 26 DIDEDUC", not fuera_didec, str(fuera_didec))

# Municipio validado contra el catálogo authoritative del CODIGO (segmento de municipio):
# cada (departamento, código-municipio) debe corresponder a un único nombre y viceversa.
_dep2 = dfc["CODIGO"].str[:2]
_mun2 = dfc["CODIGO"].str.split("-").str[1]
conf_nombre = int((dfc.assign(_d=_dep2,_m=_mun2).groupby(["_d","_m"])["MUNICIPIO"].nunique() > 1).sum())
conf_codigo = int((dfc.assign(_d=_dep2).groupby(["_d","MUNICIPIO"], observed=True)["CODIGO"]
                   .apply(lambda s: s.str.split("-").str[1].nunique()) > 1).sum())
mun_vacio = int(dfc["MUNICIPIO"].isna().sum())
check("MUNICIPIO consistente 1:1 con el código de municipio del CODIGO", conf_nombre == 0,
      f"{conf_nombre} códigos con nombres distintos")
check("Cada MUNICIPIO mapea a un único código dentro del departamento", conf_codigo == 0,
      f"{conf_codigo} nombres con códigos distintos")
check("MUNICIPIO sin valores faltantes", mun_vacio == 0, f"{mun_vacio} vacíos")

# ---- 5) Tipos de dato esperados ----
for c in CATEGORICAS:
    dfc[c] = dfc[c].astype("category")
for c in TEXTO:
    dfc[c] = dfc[c].astype("object")
    
dfc["N_TELEFONOS"] = dfc["N_TELEFONOS"].astype("int16")
dfc["ES_CIUDAD_CAPITAL"] = dfc["ES_CIUDAD_CAPITAL"].astype("bool")
dfc["DUP_PARCIAL_SOSPECHOSO"] = dfc["DUP_PARCIAL_SOSPECHOSO"].astype("bool")
dfc["TELEFONO_VALIDO"] = dfc["TELEFONO_VALIDO"].astype("boolean")

# Realizamos la prueba
tipos_ok = (
    all(ptypes.is_object_dtype(dfc[c]) for c in TEXTO) and
    all(isinstance(dfc[c].dtype, pd.CategoricalDtype) for c in CATEGORICAS) and
    ptypes.is_integer_dtype(dfc["N_TELEFONOS"]) and
    ptypes.is_bool_dtype(dfc["ES_CIUDAD_CAPITAL"]) and
    ptypes.is_bool_dtype(dfc["DUP_PARCIAL_SOSPECHOSO"]) and
    ptypes.is_bool_dtype(dfc["TELEFONO_VALIDO"])
)
check("Cada variable tiene el tipo de dato esperado", tipos_ok)

# ---- 6) Sin categorías duplicadas por diferencias de escritura ----
def _canon(x):
    return sin_tildes(str(x)).upper().strip()
dup_escritura = {}
for c in ["SECTOR","AREA","STATUS","MODALIDAD","JORNADA","PLAN","DEPARTAMENTO","DEPARTAMENTAL",
          "DEPARTAMENTO_OFICIAL"]:
    vals = dfc[c].dropna().astype(str).unique()
    if len({_canon(v) for v in vals}) != len(set(vals)):
        dup_escritura[c] = list(vals)
check("Sin categorías duplicadas por escritura (mayúsculas/tildes/espacios)",
      not dup_escritura, str(dup_escritura))

# ---- 7) Sin valores inválidos detectados en el diagnóstico ----
PH_RESTANTES = 0
for c in dfc.columns:
    if str(dfc[c].dtype) in ("category","object"):
        PH_RESTANTES += int(dfc[c].dropna().astype(str).str.strip().str.lower().isin(PLACEHOLDERS).sum())
area_mal = int((dfc["AREA"].astype(str) == "SIN ESPECIFICAR").sum())
dist_mal = int(dfc["DISTRITO"].dropna().str.match(r"^\d{2}-$").sum())
fecha_en_dir = int(dfc["DIRECCION"].dropna().str.contains(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", regex=True).sum())
comillas = int(dfc["ESTABLECIMIENTO"].dropna().str.match(r'^".*"$').sum())
cod_mal = int((~dfc["CODIGO"].fillna("").str.match(r"^\d{2}-\d{2}-\d{4}-\d{2}$")).sum())
check("Sin marcadores de ausencia residuales (--, -, N/A, etc.)", PH_RESTANTES == 0, f"{PH_RESTANTES} celdas")
check("AREA sin 'SIN ESPECIFICAR'", area_mal == 0, f"{area_mal} filas")
check("DISTRITO sin códigos truncos '##-'", dist_mal == 0, f"{dist_mal} filas")
check("DIRECCION sin fechas incrustadas", fecha_en_dir == 0, f"{fecha_en_dir} filas")
check("ESTABLECIMIENTO sin comillas envolventes", comillas == 0, f"{comillas} filas")
check("CODIGO con formato válido ##-##-####-##", cod_mal == 0, f"{cod_mal} filas")

reporte = pd.DataFrame(resultados)
reporte

,prueba,estado,detalle
0,Sin duplicados exactos (fila completa),PASS,0 duplicados
1,CODIGO es clave única,PASS,0 códigos repetidos
2,Sin espacios inicio/final/dobles ni invisibles,PASS,columnas con problema: {}
3,TELEFONO_PRINCIPAL: exactamente 8 dígitos,PASS,0 inválidos
4,TELEFONO_ADICIONALES: cada número de 8 dígitos,PASS,0 inválidos
5,N_TELEFONOS coincide con los teléfonos almacenados,PASS,
6,DEPARTAMENTO_OFICIAL en catálogo de 22 departamentos,PASS,set()
7,DEPARTAMENTO (portal) en catálogo (+ CIUDAD CAPITAL),PASS,set()
8,DEPARTAMENTAL en catálogo de 26 DIDEDUC,PASS,set()
9,MUNICIPIO consistente 1:1 con el código de municipio del...,PASS,0 códigos con nombres distintos


In [28]:
n_fail = (reporte["estado"] == "FAIL").sum()
print(f"Pruebas ejecutadas: {len(reporte)}  |  PASS: {(reporte['estado']=='PASS').sum()}  |  FAIL: {n_fail}")
if n_fail:
    print("\nFALLIDAS:")
    print(reporte[reporte['estado']=='FAIL'].to_string(index=False))
assert n_fail == 0, f"{n_fail} prueba(s) de validación fallaron: el conjunto NO está listo."
print("\n✔ Todas las pruebas de validación pasaron: el conjunto limpio está listo para el análisis.")

Pruebas ejecutadas: 20  |  PASS: 20  |  FAIL: 0

✔ Todas las pruebas de validación pasaron: el conjunto limpio está listo para el análisis.


## 8. Informe de calidad de los datos

A continuación se presenta la comparación del estado del conjunto de datos antes de iniciar las operaciones y después de ejecutar el proceso de limpieza y validación.

In [32]:
from IPython.display import display, Markdown

# Cálculo de métricas dinámicas para el informe de calidad
n_regs = len(dfc)
n_vars_antes = df.shape[1]
n_vars_despues = dfc.shape[1]
n_nas_despues = dfc.isna().sum().sum()
vars_na_antes = (faltante_mask.sum() > 0).sum()
vars_na_despues = (dfc.isna().sum() > 0).sum()
dup_parciales = dfc['DUP_PARCIAL_SOSPECHOSO'].sum()

informe_md = f"""
## 8. Informe de calidad de los datos

A continuación se presenta la comparación del estado del conjunto de datos antes de iniciar las operaciones y después de ejecutar el proceso de limpieza y validación.

| Métrica | Antes | Después |
| :--- | :--- | :--- |
| **Registros** | {len(df):,} | {n_regs:,} |
| **Variables** | {n_vars_antes} | {n_vars_despues} |
| **Valores faltantes** | {n_ph_total:,} | {n_nas_despues:,} (NAs reales) |
| **Variables con NA** | {vars_na_antes} | {vars_na_despues} |
| **Duplicados exactos** | {df.duplicated().sum()} | {dfc.duplicated().sum()} |
| **Posibles duplicados** | No identificados | {dup_parciales:,} marcados |
| **Variables con formato inconsistente** | Múltiples (fechas en dirección, teléfonos mixtos) | 0 |
| **Variables con tipo incorrecto** | {n_vars_antes} (Cargadas como texto) | 0 (Tipos asignados) |
| **Categorías inconsistentes** | Múltiples (tildes, diferencias de escritura) | 0 |
| **Errores corregidos** | N/A | Miles de unificaciones automáticas |

### Justificación de las métricas

*   **Registros:** El total se mantuvo en {n_regs:,} al no existir registros idénticos en todas las columnas. Los {dup_parciales:,} casos de posibles duplicados parciales por ejemplo mismo establecimiento, diferente plan o jornada, se conservaron y marcaron mediante la bandera lógica `DUP_PARCIAL_SOSPECHOSO` para evitar la pérdida de información legítima.
*   **Variables:** El incremento de {n_vars_antes} a {n_vars_despues} columnas responde a la creación de variables derivadas. Esto permitió aislar los datos estructurados sin sobreescribir ni destruir la trazabilidad de las columnas originales crudas.
*   **Valores faltantes y Variables con NA:** El aumento en el número de valores nulos y variables no representa una pérdida de datos, sino una ganancia en calidad. Se estandarizaron los marcadores (`--`, `SIN DATO`, etc.) a verdaderos nulos computacionales (`NaN`). Segundo, la creación de variables derivadas generó nulos estructurales lógicos.
"""

display(Markdown(informe_md))


## 8. Informe de calidad de los datos

A continuación se presenta la comparación del estado del conjunto de datos antes de iniciar las operaciones y después de ejecutar el proceso de limpieza y validación.

| Métrica | Antes | Después |
| :--- | :--- | :--- |
| **Registros** | 11,867 | 11,867 |
| **Variables** | 17 | 25 |
| **Valores faltantes** | 4,141 | 29,818 (NAs reales) |
| **Variables con NA** | 6 | 11 |
| **Duplicados exactos** | 0 | 0 |
| **Posibles duplicados** | No identificados | 1,025 marcados |
| **Variables con formato inconsistente** | Múltiples (fechas en dirección, teléfonos mixtos) | 0 |
| **Variables con tipo incorrecto** | 17 (Cargadas como texto) | 0 (Tipos asignados) |
| **Categorías inconsistentes** | Múltiples (tildes, diferencias de escritura) | 0 |
| **Errores corregidos** | N/A | Miles de unificaciones automáticas |

### Justificación de las métricas

*   **Registros:** El total se mantuvo en 11,867 al no existir registros idénticos en todas las columnas. Los 1,025 casos de posibles duplicados parciales por ejemplo mismo establecimiento, diferente plan o jornada, se conservaron y marcaron mediante la bandera lógica `DUP_PARCIAL_SOSPECHOSO` para evitar la pérdida de información legítima.
*   **Variables:** El incremento de 17 a 25 columnas responde a la creación de variables derivadas. Esto permitió aislar los datos estructurados sin sobreescribir ni destruir la trazabilidad de las columnas originales crudas.
*   **Valores faltantes y Variables con NA:** El aumento en el número de valores nulos y variables no representa una pérdida de datos, sino una ganancia en calidad. Se estandarizaron los marcadores (`--`, `SIN DATO`, etc.) a verdaderos nulos computacionales (`NaN`). Segundo, la creación de variables derivadas generó nulos estructurales lógicos.
